## 随机化AE并frozen VLM （PI05）

**my_from_pretrained** 加载预训练 VLM 权重并冻结；Action Expert 与 action/time 投影层保持随机初始化

In [1]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"  # 禁止联网，只用本地 cache

from lerobot.datasets.lerobot_dataset import LeRobotDataset

dataset = LeRobotDataset(
    repo_id='/vla/.data/test',
    video_backend='torchcodec',
)
dataset

/mnt/workspace/luyi/.cache/miniconda3/envs/myvla/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LeRobotDataset({
    Repository ID: '/vla/.data/test',
    Number of selected episodes: '1',
    Number of selected samples: '436',
    Features: '['observation.state', 'action', 'observation.images.robot0_agentview_left_image', 'observation.images.robot0_agentview_right_image', 'observation.images.robot0_eye_in_hand_image', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})',

### 模型加载

In [2]:
import torch
from lerobot.utils.import_utils import register_third_party_plugins
from lerobot.policies.factory import make_pre_post_processors

from lerobot_policy_pi05 import PI05Policy as My_PI05Policy
from lerobot.policies.pi05 import PI05Policy as Official_PI05Policy

model_id = "/vla/.models/lerobot-pi05_base"
register_third_party_plugins()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

my_policy = My_PI05Policy.my_from_pretrained(model_id).to(device).eval() # 默认冻结VLM
official_policy = Official_PI05Policy.from_pretrained(model_id).to(device).eval()

/mnt/workspace/luyi/.cache/miniconda3/envs/myvla/lib/python3.10/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loading VLM weights from: /vla/.models/lerobot-pi05_base
✓ Loaded 603 VLM tensors; AE and projections kept random init
  Missing keys (expected for AE/projections): 210


The PI05 model is a direct port of the OpenPI implementation. 
This implementation follows the original OpenPI structure for compatibility. 
Original implementation: https://github.com/Physical-Intelligence/openpi


Loading model from: /vla/.models/lerobot-pi05_base
✓ Loaded state dict from model.safetensors
Remapped: action_in_proj.bias -> model.action_in_proj.bias
Remapped: action_in_proj.weight -> model.action_in_proj.weight
Remapped: action_out_proj.bias -> model.action_out_proj.bias
Remapped: action_out_proj.weight -> model.action_out_proj.weight
Remapped: paligemma_with_expert.gemma_expert.lm_head.weight -> model.paligemma_with_expert.gemma_expert.lm_head.weight
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.bias -> model.paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.bias
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.weight -> model.paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.weight
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.mlp.down_proj.weight -> model.paligemma_with_expert.gemma_expert.model.layers.0.mlp.down_proj.weight
Remapped: paligemma_with_exp

In [3]:
my_preprocess, my_postprocess = make_pre_post_processors(
    my_policy.config,
    dataset_stats=dataset.meta.stats,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)

official_preprocess, official_postprocess = make_pre_post_processors(
    official_policy.config,
    dataset_stats=dataset.meta.stats,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)

### 推理验证
验证模型具备正常推理能力

In [4]:
def to_device(batch):
    return {
        k: v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v
        for k, v in batch.items()
    }

batch = to_device(my_preprocess(dataset[0]))

In [5]:
with torch.inference_mode():
    pred_action_raw = my_policy.select_action(batch)
pred_action_raw

tensor([[-0.3116, -0.6947,  0.1529,  0.2131,  2.0064, -1.3666,  0.8218, -0.9853,
         -1.4043,  0.5449, -0.1513, -0.0454, -0.4827, -0.0683,  1.4083, -0.6334,
         -0.1568,  1.5268, -1.2238,  1.6620, -1.8481, -1.5527,  0.2091, -2.3172,
          1.0398,  0.0286, -0.6036,  0.5831,  0.8127,  0.6018, -0.2557,  0.3084]],
       device='cuda:0')

In [6]:
batch = to_device(official_preprocess(dataset[0]))
with torch.inference_mode():
    pred_action_raw = official_policy.select_action(batch)
pred_action_raw

tensor([[-5.2361e-01,  9.6033e-01,  4.6237e-01,  8.9004e-01,  5.2967e-01,
         -4.4799e-01, -9.4198e-01,  1.8077e-01,  1.5303e-02,  6.8089e-03,
          7.1964e-03,  6.1340e-04,  7.4083e-02,  8.3916e-02,  4.9710e-01,
          5.8618e-01, -2.0538e-02,  9.2581e-02,  8.5156e-02, -9.0757e-02,
          1.9957e-01,  1.4241e-01, -3.6320e-02,  2.0120e-01,  3.6885e-02,
         -6.6484e-02,  6.9624e-02, -5.4288e-02, -1.0562e-02,  2.9064e-02,
         -8.4722e-02,  6.3756e-02]], device='cuda:0')

### 非等价验证

正确对比方式（每步输出 PASS / FAIL）：
1. 比权重 state_dict，VLM部分权重相同，但是AE部分权重不同 
2. 验证冻结部分，official 全程无冻结; my_policy的VLM部分冻结，其他部分不冻结
3. 比 predict_action_chunk（固定 noise）,输出不同值

In [7]:
from lerobot.datasets.factory import resolve_delta_timestamps

print("1/3 resolve_delta_timestamps ...")
delta_ts = resolve_delta_timestamps(my_policy.config, dataset.meta)
print("   delta_ts keys:", list(delta_ts.keys()) if delta_ts else None)

print("2/3 构建 dataset_train（含 chunk action）...")
dataset_train = LeRobotDataset(
    repo_id="/vla/.data/test",
    delta_timestamps=delta_ts,
    video_backend="torchcodec",
)

print("3/3 读取 dataset_train[0] + preprocess（含视频解码，可能需 10~30s）...")
sample = dataset_train[0]
print("   raw action shape:", sample["action"].shape)  # 期望 [50, 12]

batch_eq = to_device(my_preprocess(sample))

# preprocess 有时不会给 chunk action 加 batch 维，需手动 [50,D] -> [1,50,D]
if batch_eq["action"].ndim == 2:
    batch_eq["action"] = batch_eq["action"].unsqueeze(0)

action_dim = my_policy.config.output_features["action"].shape[0]
print("batch_eq action shape:", batch_eq["action"].shape)  # 期望 [1, 50, 12]

1/3 resolve_delta_timestamps ...
   delta_ts keys: ['action']
2/3 构建 dataset_train（含 chunk action）...
3/3 读取 dataset_train[0] + preprocess（含视频解码，可能需 10~30s）...
   raw action shape: torch.Size([50, 12])
batch_eq action shape: torch.Size([1, 50, 12])


#### Step 1 — 比权重（VLM 相同 / AE 不同）

In [8]:
VLM_PREFIX = "paligemma_with_expert.paligemma."
AE_PREFIXES = (
    "paligemma_with_expert.gemma_expert.",
    "action_in_proj.",
    "action_out_proj.",
    "time_mlp_in.",
    "time_mlp_out.",
)


def _max_abs_diff(t1, t2):
    return (t1.float().cpu() - t2.float().cpu()).abs().max().item()


def compare_weights_non_equiv(m1, m2, atol=0.0):
    s1, s2 = m1.model.state_dict(), m2.model.state_dict()
    common = sorted(set(s1) & set(s2))

    vlm_keys = [k for k in common if k.startswith(VLM_PREFIX)]
    ae_keys = [k for k in common if any(k.startswith(p) for p in AE_PREFIXES)]

    vlm_mismatch = [
        (k, _max_abs_diff(s1[k], s2[k]))
        for k in vlm_keys
        if _max_abs_diff(s1[k], s2[k]) > atol
    ]
    ae_match = [
        (k, _max_abs_diff(s1[k], s2[k]))
        for k in ae_keys
        if _max_abs_diff(s1[k], s2[k]) <= atol
    ]

    vlm_ok = len(vlm_mismatch) == 0
    ae_ok = len(ae_match) == 0
    ok = vlm_ok and ae_ok

    print("=" * 50)
    print("Step 1: 权重对比（期望 VLM 相同 / AE 不同）")
    print(f"  VLM keys: {len(vlm_keys)} | AE keys: {len(ae_keys)}")
    print(f"  VLM 不一致 ({len(vlm_mismatch)}):", [k for k, _ in vlm_mismatch[:3]], "..." if len(vlm_mismatch) > 3 else "")
    print(f"  AE  仍相同 ({len(ae_match)}):", [k for k, _ in ae_match[:3]], "..." if len(ae_match) > 3 else "")
    if vlm_mismatch:
        print(f"  VLM 最大 diff: {vlm_mismatch[0][0]} = {vlm_mismatch[0][1]:.6e}")
    if ae_match:
        print(f"  AE  相同示例: {ae_match[0][0]} diff={ae_match[0][1]:.6e}")
    print("  >>>", "PASS ✓" if ok else "FAIL ✗")
    return ok


step1_ok = compare_weights_non_equiv(my_policy, official_policy)

Step 1: 权重对比（期望 VLM 相同 / AE 不同）
  VLM keys: 604 | AE keys: 209
  VLM 不一致 (0): [] 
  AE  仍相同 (0): [] 
  >>> PASS ✓


#### Step 2 — 验证冻结状态

In [9]:
def freeze_summary(policy):
    pge = policy.model.paligemma_with_expert
    vlm_grad = [p.requires_grad for p in pge.paligemma.parameters()]
    ae_grad = [p.requires_grad for p in pge.gemma_expert.parameters()]
    proj_names = ["action_in_proj", "action_out_proj", "time_mlp_in", "time_mlp_out"]
    proj_grad = {n: getattr(policy.model, n).weight.requires_grad for n in proj_names}
    return {
        "vlm_trainable": sum(vlm_grad),
        "vlm_total": len(vlm_grad),
        "ae_trainable": sum(ae_grad),
        "ae_total": len(ae_grad),
        "proj_grad": proj_grad,
    }


def check_freeze_non_equiv(my_policy, official_policy):
    my_s = freeze_summary(my_policy)
    off_s = freeze_summary(official_policy)

    off_ok = (
        my_s["vlm_total"] == off_s["vlm_total"]
        and off_s["vlm_trainable"] == off_s["vlm_total"]
        and off_s["ae_trainable"] == off_s["ae_total"]
        and all(off_s["proj_grad"].values())
    )
    my_ok = (
        my_s["vlm_trainable"] == 0
        and my_s["ae_trainable"] == my_s["ae_total"]
        and all(my_s["proj_grad"].values())
    )
    ok = off_ok and my_ok

    print("=" * 50)
    print("Step 2: 冻结状态（期望 official 全可训 / my 仅 VLM 冻结）")
    print(f"  official VLM trainable: {off_s['vlm_trainable']}/{off_s['vlm_total']}")
    print(f"  official AE  trainable: {off_s['ae_trainable']}/{off_s['ae_total']}")
    print(f"  official proj trainable: {off_s['proj_grad']}")
    print(f"  my       VLM trainable: {my_s['vlm_trainable']}/{my_s['vlm_total']}")
    print(f"  my       AE  trainable: {my_s['ae_trainable']}/{my_s['ae_total']}")
    print(f"  my       proj trainable: {my_s['proj_grad']}")
    print("  >>>", "PASS ✓" if ok else "FAIL ✗")
    return ok


step2_ok = check_freeze_non_equiv(my_policy, official_policy)

Step 2: 冻结状态（期望 official 全可训 / my 仅 VLM 冻结）
  official VLM trainable: 603/603
  official AE  trainable: 201/201
  official proj trainable: {'action_in_proj': True, 'action_out_proj': True, 'time_mlp_in': True, 'time_mlp_out': True}
  my       VLM trainable: 0/603
  my       AE  trainable: 201/201
  my       proj trainable: {'action_in_proj': True, 'action_out_proj': True, 'time_mlp_in': True, 'time_mlp_out': True}
  >>> PASS ✓


#### Step 3 — 比 predict_action_chunk（固定 noise）

In [10]:
cfg = my_policy.config
fixed_noise = torch.randn(1, cfg.chunk_size, cfg.max_action_dim, device=device, dtype=torch.float32)

with torch.inference_mode():
    chunk_my = my_policy.predict_action_chunk(batch_eq, noise=fixed_noise.clone())
    chunk_off = official_policy.predict_action_chunk(batch_eq, noise=fixed_noise.clone())

action_my = chunk_my[0, 0, :action_dim]
action_off = chunk_off[0, 0, :action_dim]
action_diff = (action_my - action_off).abs().max().item()
step3_ok = action_diff < 1e-4

print("=" * 50)
print("Step 3: predict_action_chunk（固定 noise）")
print(f"  action[0,:3] my      = {action_my[:3].tolist()}")
print(f"  action[0,:3] official= {action_off[:3].tolist()}")
print(f"  max |diff|           = {action_diff:.2e}")
print("  >>>", "PASS ✓" if step3_ok else "FAIL ✗")

Step 3: predict_action_chunk（固定 noise）
  action[0,:3] my      = [-0.8663476705551147, -1.956830620765686, 1.5008344650268555]
  action[0,:3] official= [-0.5251425504684448, 1.0247784852981567, 0.49370691180229187]
  max |diff|           = 2.98e+00
  >>> FAIL ✗


#### 汇总

In [11]:
step3_non_equiv_ok = action_diff > 1e-4

results = {
    "Step1 权重(VLM同/AE异)": step1_ok,
    "Step2 冻结状态": step2_ok,
    "Step3 推理(chunk)不同": step3_non_equiv_ok,
}
print("=" * 50)
print("汇总")
for name, ok in results.items():
    print(f"  {name}: {'PASS ✓' if ok else 'FAIL ✗'}")
print(f"  Step3 max |diff| = {action_diff:.2e}")
print("=" * 50)
print("整体:", "全部 PASS ✓ — 非等价验证通过" if all(results.values()) else "存在 FAIL ✗ — 见上方详情")

if not step1_ok:
    print("\n提示: Step1 失败时检查 my_from_pretrained 是否只加载了 VLM 权重。")
if not step2_ok:
    print("\n提示: Step2 失败时检查 config.train_expert_only 是否为 True。")
if not step3_non_equiv_ok:
    print("\n提示: Step3 失败说明两 policy 推理输出仍相同，AE 可能未被随机化。")

汇总
  Step1 权重(VLM同/AE异): PASS ✓
  Step2 冻结状态: PASS ✓
  Step3 推理(chunk)不同: PASS ✓
  Step3 max |diff| = 2.98e+00
整体: 全部 PASS ✓ — 非等价验证通过


随机化模型存储

In [ ]:
from pathlib import Path
save_dir = Path("/vla/.models/pi05_AE_random")
save_dir.mkdir(parents=True, exist_ok=True)
# 1. 保存模型权重 + config.json（约 14GB）
my_policy.save_pretrained(save_dir)
# 2. 可选：保存 preprocessor（推理时需要与训练一致的归一化 stats）
my_preprocess.save_pretrained(save_dir, config_filename="policy_preprocessor.json")
my_postprocess.save_pretrained(save_dir, config_filename="policy_postprocessor.json")

: 